## import

In [ ]:
import re, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

sns.set_theme(style="whitegrid")
random.seed(42); np.random.seed(42)

## bikin dataset chat sintetis

In [ ]:
SIGNAL = {
    "invite":   ["besok jadi nonton?", "mau makan bareng ga?", "weekend ada acara ga?", "ayo ketemu minggu ini", "kapan kita jalan?"],
    "engage":   ["terus gimana tadi?", "kamu sendiri gimana?", "ceritain dong", "lah kok bisa?", "emang kenapa sih?"],
    "brush_off":["nanti ya lagi rapat", "gabisa lagi sibuk", "ntar aja deh", "lagi ga bisa", "next time aja ya"],
    "none":     [],
}
BY_EXCITE = {
    "high":   ["seneng banget sih", "akhirnya jadi juga", "gabisa nunggu buat besok", "keren banget", "serius mau banget", "asik ayo berangkat"],
    "medium": ["boleh juga", "oke bisa kok", "lumayan seru", "kayaknya bisa", "nanti aku kabarin ya"],
    "low":    ["hm", "yaudah", "terserah", "iya", "oke", "lagi capek", "gitu ya"],
}
EMOJI = ["😂", "🥰", "😍", "🔥", "😭", "🤩", "😊"]

def stretch(w):
    return re.sub(r"([aiueo])(?=[^aiueo]*$)", lambda m: m.group(1) * random.randint(3, 5), w, count=1)

def decorate(text, excite, hard=False):
    # profil penanda: seberapa sering tanda seru, emoji, tawa, kata dipanjangin
    p = {"high": (0.8, 0.6, 0.4, 0.4), "medium": (0.3, 0.25, 0.2, 0.1), "low": (0.05, 0.05, 0.05, 0.0)}[excite]
    if hard:  # kasus sulit: penanda sengaja tidak sesuai label
        p = {"high": (0.05, 0.05, 0.05, 0.0), "medium": (0.6, 0.5, 0.3, 0.2), "low": (0.7, 0.5, 0.3, 0.3)}[excite]
    ex, em, lg, st = p
    words = text.split()
    if random.random() < st and words:
        words[-1] = stretch(words[-1])
    text = " ".join(words)
    if random.random() < lg: text += random.choice([" wkwk", " wkwkwk", " haha", " hehe"])
    if random.random() < ex: text += "!" * random.randint(1, 3)
    if random.random() < em: text += " " + random.choice(EMOJI) * random.randint(1, 2)
    return text

def make_dataset(n=2000):
    rows = []
    for i in range(n):
        signal = random.choices(list(SIGNAL), weights=[0.15, 0.2, 0.15, 0.5])[0]
        excite = random.choices(["low", "medium", "high"], weights=[0.35, 0.35, 0.3])[0]
        base = random.choice(SIGNAL[signal]) if signal != "none" else random.choice(BY_EXCITE[excite])
        hard = random.random() < 0.12
        rows.append({"msg_id": i + 1, "sender": random.choice(["A", "B"]),
                     "text": decorate(base, excite, hard), "excitement": excite, "signal": signal, "hard_case": hard})
    return pd.DataFrame(rows)

df = make_dataset(2000)
df.to_csv("chat_dataset_sintetis.csv", index=False)
print(df.shape); df.sample(8, random_state=1)

## EDA

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(x="excitement", hue="excitement", data=df, order=["low", "medium", "high"], palette="Set2", legend=False, ax=ax[0])
ax[0].set_title("Distribusi label excitement")
sns.countplot(x="signal", hue="signal", data=df, palette="Pastel1", legend=False, ax=ax[1])
ax[1].set_title("Distribusi label signal")
plt.show()

## heuristik excited score (SAMA dengan yang dipakai di aplikasi)

In [ ]:
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\u2600-\u27BF]")
LAUGH_RE = re.compile(r"\b(w[k]{2,}|wk(wk)+|haha+|hehe+|xixi+|awok+)\b", re.I)
STRETCH_RE = re.compile(r"([a-zA-Z])\1{2,}")

def excited_score(text):
    ex = min(text.count("!"), 3)
    em = min(len(EMOJI_RE.findall(text)), 3)
    lg = min(len(LAUGH_RE.findall(text)), 2)
    st = min(len(STRETCH_RE.findall(text)), 2)
    score = ex * 14 + em * 12 + lg * 10 + st * 12   # maks sekitar 100
    return min(score, 100)

df["score"] = df["text"].apply(excited_score)
sns.boxplot(x="excitement", y="score", hue="excitement", data=df, order=["low", "medium", "high"], palette="Set2", legend=False)
plt.title("Excited score per label"); plt.show()

# cari ambang batas terbaik di data training, uji di data testing
train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["excitement"])
def to_class(s, t1, t2): return "low" if s < t1 else ("medium" if s < t2 else "high")
best = max(((t1, t2) for t1 in range(5, 60, 5) for t2 in range(t1 + 5, 100, 5)),
           key=lambda t: accuracy_score(train["excitement"], train["score"].apply(lambda s: to_class(s, *t))))
print("Ambang terbaik (low/medium, medium/high):", best)
pred = test["score"].apply(lambda s: to_class(s, *best))
print(f"Akurasi heuristik (data testing): {accuracy_score(test['excitement'], pred)*100:.2f}%")
print(classification_report(test["excitement"], pred))

## model machine learning

In [ ]:
Xtr, Xte, ytr, yte = train["text"], test["text"], train["excitement"], test["excitement"]
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=True)
Xtr_v, Xte_v = tfidf.fit_transform(Xtr), tfidf.transform(Xte)

lr = LogisticRegression(max_iter=3000, random_state=42).fit(Xtr_v, ytr)
print(f"Akurasi Regresi Logistik (TF-IDF): {accuracy_score(yte, lr.predict(Xte_v))*100:.2f}%")

rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr_v, ytr)
print(f"Akurasi Random Forest (TF-IDF): {accuracy_score(yte, rf.predict(Xte_v))*100:.2f}%")
print(classification_report(yte, lr.predict(Xte_v)))

cm = confusion_matrix(yte, lr.predict(Xte_v), labels=["low", "medium", "high"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", xticklabels=["low", "medium", "high"], yticklabels=["low", "medium", "high"])
plt.title("Confusion matrix (Regresi Logistik)"); plt.xlabel("Prediksi"); plt.ylabel("Label"); plt.show()

## model untuk label signal (invite / engage / brush_off / none)

In [ ]:
ytr_s, yte_s = train["signal"], test["signal"]
lr_s = LogisticRegression(max_iter=3000, random_state=42).fit(Xtr_v, ytr_s)
print(f"Akurasi signal (Regresi Logistik): {accuracy_score(yte_s, lr_s.predict(Xte_v))*100:.2f}%")
print(classification_report(yte_s, lr_s.predict(Xte_v)))

## analisis error

In [ ]:
err = test.assign(pred=pred)[lambda d: d["pred"] != d["excitement"]]
print(f"Jumlah salah tebak heuristik: {len(err)} dari {len(test)}")
err[["text", "excitement", "pred", "score", "hard_case"]].head(10)